# CSP-basierte Verteilung zentraler Prüfungsräume

```Christopher Klein & Colin Vavra```

Für die Prüfungswochen der Hochschule werden **zwei große Prüfungsräume** benötigt.

**Mehrere Studiengänge** greifen in denselben Wochen auf diese Räume zu. Die verfügbaren **Raum-Zeit**-Slots sollen so verteilt werden, dass die **unterschiedlichen Bedarfe** der Studiengänge berücksichtigt werden und gleichzeitig **organisatorisch sinnvolle** sowie **faire Belegungspläne** entstehen.

Es wird **keine Zielfunktion** optimiert, gesucht wird eine *zulässige* Belegung, die alle formulierten Constraints erfüllt.

**Struktur des Notebooks**
1. Problemanalyse und formale CSP-Modellierung
2. Ableitung der Bedarfs- und Fairness-Constraints
3. Implementierung des Constraint-Modells
4. Lösungen der Prüfungswochen und Validierung
5. Analyse des Lösungsraums
6. Experiment mit verändertem Constraint
7. Unlösbarkeit und Konfliktanalyse
8. Analyse des Suchverfahrens
9. Kritische Reflexion
10. Dokumentation der GenAI-Nutzung
11. Fazit

## 1. Problemanalyse und formale CSP-Modellierung

### 1.1 Entscheidungen des Planungssystems
Für jede Prüfungswoche muss entschieden werden, **welcher Studiengang** jeden der 40 Raum-Zeit-Slots (5 Tage × 4 Zeitslots × 2 Räume) belegt. Es ist stets genau ein Studiengang pro Slot zuzuweisen.

Dabei sind mithilfe von Constraints verschiedene Rahmenbedingungen zu berücksichtigen.


### 1.2 Formale CSP-Definition

```Eine mögliche Modellierung besteht beispielsweise darin, jeder Kombination aus Wochentag, Zeitslot und Raum eine Variable zuzuordnen, deren Wert den zugeordneten Studiengang angibt. Andere fachlich geeignete Repräsentationen sind zulässig.```

$$CSP = (X, D, C)$$

#### Variables/Domains

- $X_{v,s} \in \{0,1\}$
- $v = Tag\ +\ "\_"\ +\ Slot\ +\ "\_"\ +\ Raum$
- $s \in \{ "MB", "INF", ... \}$

Beispiel: ``` X["Mo_A_R1", "MB"] = 0/1```


### 1.3 Organisatorische Anforderungen als Constraints
- **Slot-Eindeutigkeit:** Jeder Raum-Zeit-Slot wird genau einem Studiengang zugewiesen.
- **Wochenbeschränkung:** Nur in der Woche beteiligte Studiengänge erhalten Slots.
- **Bedarfsgerechtigkeit:** Slot-Anspruch pro Woche anteilig gemäß Studiengangsgröße.
- **Tagesverteilung:** Kein Studiengang konzentriert seine Slots auf einen Tag (analog Woche).
- **Popularitäts-Fairness:** Mittlere Popularität pro Slot liegt nahe am globalen Mittel.
- **Parallel / Sequenziell:** Berücksichtigung der Slot-Struktur je nach gruppenabhängiger Präferenz.
- **Kein später→früher-Übergang:** Wer am Vortag den letzten Slot hat, darf am Folgetag nicht den ersten Slot belegen.

Details dazu gibt es in Abschnitt #2.


### 1.4 Slot-lokale vs. koppelnde Constraints

|Typ|Constraints|
|---|---|
| **Slot-lokal** (betreffen einzelne Variablen / einen Slot) | Slot-Eindeutigkeit, Wochenbeschränkung, parallele Belegung eines einzelnen Slots |
| **Koppelnd** (verbinden mehrere Zuordnungen) | Bedarfsgrenzen, Tagesverteilung, Popularitäts-Fairness, Parallel-Untergrenzen, Übergangsregel |
    
Die koppelnden Constraints sind fachlich die interessanteren, weil sie die **Verteilung über die ganze Woche** steuern.




### Daten

#### exam_weeks.csv
- Woche 1-3
- Studiengang
- Anzahl Gruppen

#### preferences.json
- Time Slot Preference (1-5)
- Weekday Preference (1-5)
- Structure Preference
  - 3 Preferences
  - Jede enthält
    - condition gegen Anzahl course_groups
    - Parallel vs Sequentiell Preference
    - Preferenzwert + allowed-Boolean

### Wochenplan lesen

In [29]:
import pandas as pd

exam_weeks = pd.read_csv('./data/exam_weeks.csv', delimiter=';')
exam_weeks.columns = ['exam_week','study_program','course_groups','cohorts']
#exam_weeks.head()
print(exam_weeks.to_string())

   exam_week study_program  course_groups  cohorts
0    Woche 1            ES              3        1
1    Woche 1           INF              1        1
2    Woche 1            MB              8        1
3    Woche 1            MT              4        1
4    Woche 1            DS              4        1
5    Woche 2            ET              2        1
6    Woche 2            MB              8        1
7    Woche 2            MT              4        1
8    Woche 2            DS              4        1
9    Woche 3           WIW              5        1
10   Woche 3            ET              2        1
11   Woche 3            MB              8        1
12   Woche 3            DS              4        1


### Präferenzen lesen

In [30]:
import json

with open('./data/preferences.json') as f:
    prefs = json.load(f)

# Popularität der Zeitslots (A-D) und Wochentage
print("Zeitslots:", {t: v["preference"] for t, v in prefs["time_slot_preferences"].items()})
print("Wochentage:", prefs["weekday_preferences"])

# Regeln für parallele / aufeinanderfolgende Termine
print("\nStruktur-Regeln:")
for rule in prefs["appointment_structure_preferences"]:
    print(f'  {rule["condition"]}; parallel: {rule["parallel"]},  aufeinanderfolgend: {rule["sequential"]}')

Zeitslots: {'A': 4, 'B': 5, 'C': 2, 'D': 2}
Wochentage: {'Montag': 5, 'Dienstag': 4, 'Mittwoch': 4, 'Donnerstag': 3, 'Freitag': 1}

Struktur-Regeln:
  course_groups >= 3; parallel: {'preference': 6, 'allowed': True},  aufeinanderfolgend: {'preference': 3, 'allowed': True}
  course_groups == 2; parallel: {'preference': 0, 'allowed': True},  aufeinanderfolgend: {'preference': 0, 'allowed': True}
  course_groups == 1; parallel: {'preference': 0, 'allowed': False},  aufeinanderfolgend: {'preference': 0, 'allowed': False}


In [31]:
!pip install ortools

### Init

In [33]:
import math

# Mengen definieren
weekdays = ["Mo", "Di", "Mi", "Do", "Fr"]
slots = ["A", "B", "C", "D"]
rooms = ["R1", "R2"]

# X: eine Variable für jede Kombination aus Tag, Zeitslot und Raum
variables = [f"{day}_{slot}_{room}" for day in weekdays for slot in slots for room in rooms]
N_SLOTS = len(variables)  # 40
SLOTS_PER_DAY = len(slots) * len(rooms)  # 4 Slots in jeweils 2 Räumen = 8 Slots

weeks = exam_weeks["exam_week"].unique().tolist()
print(weeks, f", Slots: {N_SLOTS}", f", Slots per Day: {SLOTS_PER_DAY}")

['Woche 1', 'Woche 2', 'Woche 3'] , Slots: 40 , Slots per Day: 8


## 2. Ableitung der Bedarfs- und Fairness-Constraints

### 2.1 Slot-Eindeutigkeit

Etablierung eines ```addExactlyOne``` Constraints über die Studiengänge pro Zeit-Slot.

### 2.2 Wochenbeschränkung

- In einer Woche dürfen nur die für diese Woche angegebenen Studiengänge Slots erhalten.
- erfüllt mittels dedizierter CSP's pro Woche

### 2.3 Bedarf

Es gilt:

- $g_s$: Anzahl der Kursgruppen des Studiengangs $s$
- $n_{\text{slots}}$: Anzahl verfügbarer Slots, z. B. 40 pro Woche oder 8 pro Tag
- $N_{\text{Woche}}$: Gesamtzahl der Kursgruppen in dieser Woche
- $n_s$: Anzahl der Slots, die dem Studiengang zugeordnet werden

#### 2.3.1 Woche

Anteil der Kursgruppen des Studiengangs in dieser Woche:
$$
q_s = \frac{g_s}{N_{\text{Woche}}}
$$

Anteilig gegen die Anzahl der Wochen-Slots rechnen:
$$bounds = 40\,q_s$$

Abrunden für Untergrenze, Aufrunden für Obergrenze.

Tatsächlicher Bedarf muss zwischen Untergrenze und Obergrenze liegen:
$$\lfloor bounds \rfloor \le n_s \le \lceil bounds \rceil$$

Beispiel Woche 1:
- MB 16
- ES 6
- MT 8
- DS 8
- INF 2

Slots für die gesamte Woche.

Rechenweg als Beispiel für "MB" dazu:
$$
q_s = 8/20 = 0.4
$$
$$
bounds = 40 * 0.4 = 16
$$

Da ganzzahlig ergibt sich sowohl für Untergrenze als auch Obergrenze der Wert **16**.


#### 2.3.2 Tag
Damit sich die Slots eines Studiengangs nicht auf einzelne Tage konzentrieren, gilt dieselbe Regel für die 8 Raum-Zeit-Slots jedes Tages (4 Zeitslots × 2 Räume):
$$\lfloor 8\,q_s \rfloor \le n_{s,d} \le \lceil 8\,q_s \rceil \quad \text{für jeden Tag } d$$

Der tägliche Bedarf ergibt sich damit, wie gefordert, aus der Größe des Studiengangs **und** aus den anderen Studiengängen derselben Woche, weil beide in $q_s$ eingehen.

Beispiel Woche 1:
- MB 3–4
- ES 1–2
- MT 1–2
- DS 1–2
- INF 0–1

Slots pro Tag.

Ist $\lfloor 8\,q_s \rfloor \ge 1$, ist der Studiengang automatisch an jedem Tag vertreten (vgl. 4.5).

`demand_bounds(week_df, n_slots)` berechnet beide Grenzen, für die Woche mit `n_slots=40`, für den Tag mit `n_slots=8`.



In [21]:
def demand_bounds(week_df, n_slots=N_SLOTS):

    # Studiengänge zu Kursgruppen
    groups = dict(zip(
        week_df["study_program"],
        week_df["course_groups"] * week_df["cohorts"]
    ))

    total_of_week = sum(groups.values()) # Gesamtbedarf aller Gruppen dieser Woche

    # Anteiliger Bedarf
    results = {}
    for s, g in groups.items(): # s = StudyProgram, g=Kursgruppen
        q_s = g / total_of_week # Anteil Gruppen des Studiengangs an allen Gruppen der Woche
        bound = q_s * n_slots # Mit diesem Quotienten die Slots anteilig ermitteln aus Gesamtslots (Woche = 40, Tag = 8)

        results[s] = (
            math.floor(bound),  # lo
            math.ceil(bound)   # hi
        )

    return results
    # ergibt Return Wert mit Dict s: (lo, hi)



In [22]:
week_df = exam_weeks[exam_weeks["exam_week"] == "Woche 1"]
print (week_df)

bounds_week = demand_bounds(week_df)
print(f"bounds_week: {bounds_week}")

bounds_day = demand_bounds(week_df, 8)
print(f"bounds_day: {bounds_day}")

  exam_week study_program  course_groups  cohorts
0   Woche 1            ES              3        1
1   Woche 1           INF              1        1
2   Woche 1            MB              8        1
3   Woche 1            MT              4        1
4   Woche 1            DS              4        1
bounds_week: {'ES': (6, 6), 'INF': (2, 2), 'MB': (16, 16), 'MT': (8, 8), 'DS': (8, 8)}
bounds_day: {'ES': (1, 2), 'INF': (0, 1), 'MB': (3, 4), 'MT': (1, 2), 'DS': (1, 2)}


### 2.4 Popularität

#### Popularitätsmaß
Tages- und Slotwert aus `preferences.json` werden addiert:
$$p(d,t) = \text{pref}_{Tag}(d) + \text{pref}_{Slot}(t)$$

(gemäß Aufgabenblatt, 4.3: ```Beispielsweise können die Werte von Tag und Zeitslot additiv kombiniert werden```)

Beispiel:
- Montag A = *5 + 4 = 9*
- Freitag D = *1 + 2 = 3*

Popularität in Woche 1 über alle 40 Slots = $266$

Damit ergibt sich die mittlere Popularität $$\bar p = 266 / 40 = 6.65$$

#### Fairness-Constraint

Man kann nun für die **Popularität der von einem Studiengang verwendeten Slots aufsummieren** und **durch die vom Studiengang belegten Slots teilen** und erhält so die gemittelte Popularität bezogen auf die Slots dieses Studiengangs.

$$\textbf{Mittlere_Popularität_der_Studiengang_Slots} = \frac{\sum_{v} p(v) * \,x_{v,s}}{n_s}$$

Dieser Wert muss sich um den ermittelten Wert von $6.65$ befinden.

Das Intervall muss allerdings weiter nach links und rechts geöffnet werden. Hierzu wird der Toleranzwert $\tau$ verwendet.

$$6.65 - \tau \;\le\; \textbf{Mittlere_Popularität_der_Studiengang_Slots} \;\le\; 6.65 + \tau$$

**Achtung!** CP-SAT kann nicht mit Brüchen rechnen. Deshalb wird die Ungleichung umgeformt:
$$\left|\sum_v (40\,p(v) - 266)\,x_{v,s}\right| \le 40\,\tau\, n_s$$

Umformung gelang durch:
- **Nenner entfernen**: Multiplizieren mit Nenner n_s
- **Zusammenfassen**: Subtrahiere pMean (6.65) + Betrag
- **Dezimalzahl beseitigen**: pMean = 266/40 = 6.65 -> Einsetzen.
  - Dann Ungleichung mit 40 multiplizieren
  - In Summe reinziehen


#### Wahl der Toleranz $\tau = 0.5$
Herleitung und Experiment dazu: siehe Abschnitt 6.


In [23]:
# Wir erstellen ein Mapping:
# Wochentage im Code ("Mo") -> Namen in preferences.json ("Montag")
day_names = {"Mo": "Montag", "Di": "Dienstag", "Mi": "Mittwoch", "Do": "Donnerstag", "Fr": "Freitag"}

# Popularität eines Slots berechnen - p(d, t)
def popularity(day, slot):
    weekday_pop = prefs["weekday_preferences"][day_names[day]]
    slot_pop = prefs["time_slot_preferences"][slot]["preference"]
    return weekday_pop + slot_pop

# Popularität für alle Slots berechnen
pop = {}

for v in variables:
    day, slot, _ = v.split("_")
    pop[v] = popularity(day, slot)

# Beispiel: {'Mo_A_R1': 9, 'Mo_A_R2': 9, 'Mo_B_R1': 10, ...}
#print(pop)

POP_TOL = 0.5   # erlaubte Abweichung der mittleren Popularität vom Wochenmittel

popTotal = sum(pop.values())
#print(f"pT: {popTotal}")

print("mittlere Popularität aller Slots:", popTotal / N_SLOTS)

popOverview = pd.DataFrame(
    {
        d: {
            t: popularity(d, t)
            for t in slots
        }
        for d in weekdays
    }
)

print("")
print("Popularität pro Tag/Slot:")

display(popOverview)

mittlere Popularität aller Slots: 6.65

Popularität pro Tag/Slot:


,Mo,Di,Mi,Do,Fr
A,9,8,8,7,5
B,10,9,9,8,6
C,7,6,6,5,3
D,7,6,6,5,3


In [24]:
def add_popularity_constraints(model, x, programs):
    #popTotal = sum(pop.values())     # 266 = Summe der Popularität aller 40 Slots # schon oben definiert
    tol = round(POP_TOL * N_SLOTS)    # 0.5 * 40 = 20 (Toleranz, ebenfalls mit 40 multipliziert)

    for s in programs:
        n_s = sum(x[v, s] for v in variables)   # Anzahl Slots des Studiengangs (lineare Summe, wie beim Bedarf)

        # sum(40 * p_v - 266) * X_v,s
        deviation = sum(
            (N_SLOTS * pop[v] - popTotal) * x[v, s]
            for v in variables
        )

        model.Add(deviation <= tol * n_s)    # nicht zu viele attraktive Slots
        model.Add(deviation >= -tol * n_s)   # nicht zu viele unattraktive Slots

### 2.5 Struktur

#### Begriffe
- **Parallel:** Ein Studiengang hat zum selben Zeitpunkt $(d,t)$ beide Räume: $x_{d,t,R1} = x_{d,t,R2} = s$.
- **Aufeinanderfolgend:** Ein Studiengang hat am selben Tag zwei direkt aufeinanderfolgende Zeitslots (A–B, B–C, C–D), egal in welchem Raum. Für die Kursgruppen zählt der zeitliche Ablauf, nicht der Raum. Den Übergang über Nacht (D → A) behandelt 4.5.

#### Regeln aus `preferences.json`
Für jeden Studiengang wird die Bedingung (`course_groups >= 3` usw.) mit seiner Gruppenzahl $g_s$ ausgewertet:

| Bedingung | Datensatz | Constraint | betrifft |
|---|---|---|---|
| $g_s = 1$ | parallel und aufeinanderfolgend `allowed: false` | beides verboten | INF (W1) |
| $g_s = 2$ | beides erlaubt, Präferenz 0 / 0 | keine Einschränkung | ET (W2, W3) |
| $g_s \ge 3$ | beides erlaubt, parallel 6 > aufeinanderfolgend 3 | (a) #parallel ≥ #aufeinanderfolgend <br> (b) #parallel $\ge \lfloor 0.5 \cdot bounds_{min} / 2 \rfloor$ <br> verhindert dass 0>=0 akzeptiert ewrden <br> fordert Hälfte des Min-Bedarfs des Studiengangs in dieser Woche ein | alle anderen |

#### Begründung für $g_s \ge 3$
Aus den Präferenzen (parallel 6 > aufeinanderfolgend 3 > 0) folgt, dass große Studiengänge einen wesentlichen Teil ihrer Slots parallel erhalten sollen (a). Der Datensatz nennt keinen konkreten Anteil. Wir fordern etwa die Hälfte der Slots als Parallelpaare (b) und fordern dazu:
$$\left\lfloor \frac{\text{min_Bedarf_Räume} \cdot 0.5}{2} \right\rfloor$$

Ein höherer Anteil (z. B. 6/9 = 2/3) ist nicht erfüllbar: In Woche 1 lassen die Tagesgrenzen für ES (6 Slots, max. 2 pro Tag) höchstens ein Parallelpaar zu.
    - ```Hier kollidieren zwei Contraints: Tagesgrenze vs Parallelwunsch```

Erkenntnis außerdem: Ob 6/3, 10/3 oder whatever: Wichtig ist nur der größere Präferenzwert für Parallelität um diese zu bevorzugen.    

#### Umsetzung in CP-SAT
Hilfsvariablen je Tag $d$, Zeitslot $t$ und Studiengang $s$:
- `par = min(x[R1], x[R2])` ist 1, wenn $s$ beide Räume hat
    - weil wenn bei einem boolean Paar (0/1, 0/1) der kleinere Wert 1 ist, dann müssen beide 1 sein
- `used = max(x[R1], x[R2])` ist 1, wenn $s$ mindestens einen Raum hat
    - weil wenn bei einem boolean Paar (0/1, 0/1) der größere Wert 1 ist, dann ist entweder einer von beiden oder beide 1
- `seq = min(used[t], used[t+1])` ist 1, wenn $s$ zwei aufeinanderfolgende Zeitslots hat
    - analog par() bzgl. Verwendung von $min$

In [25]:
PAR_SHARE = 0.5   # Mindestanteil der Slots in Parallelpaaren (für g >= 3)

def structure_rule(g):
    # passende Regel aus preferences.json zur Gruppenzahl g finden
    # condition ist ein String wie "course_groups >= 3" und wird mit eval ausgewertet
    for rule in prefs["appointment_structure_preferences"]:
        if eval(rule["condition"], {"course_groups": g}):
            return rule

for g in [1, 2, 3, 10]:
    r = structure_rule(g)
    print(f"{g} Gruppen -> parallel: {r['parallel']}  seq: {r['sequential']}")

1 Gruppen -> parallel: {'preference': 0, 'allowed': False}  seq: {'preference': 0, 'allowed': False}
2 Gruppen -> parallel: {'preference': 0, 'allowed': True}  seq: {'preference': 0, 'allowed': True}
3 Gruppen -> parallel: {'preference': 6, 'allowed': True}  seq: {'preference': 3, 'allowed': True}
10 Gruppen -> parallel: {'preference': 6, 'allowed': True}  seq: {'preference': 3, 'allowed': True}


In [26]:
def add_structure_constraints(model, x, week_df, programs):
    groups = dict(zip(week_df["study_program"], week_df["course_groups"] * week_df["cohorts"]))
    bounds_week = demand_bounds(week_df)

    # Methode wird im jeweiligen Wochenkontext gerufen

    # ... und iteriert dann über Studiengänge der Woche

    for s in programs:

        # ... extrahiert passende Struktur-Regeln
        rule = structure_rule(groups[s])
        parallel = []
        sequential = []

        # ... iteriert Tage und deren Slots

        for day in weekdays:
            used = {}
            for t in slots:

                # das hier ist der Verweis auf "x", unser Variablensystem
                # Dabei wird jeweils R1 und R2 betrachtet, sodass deren Zustand
                # im weiteren Verlauf herangezogen werden kann.
                r1 = x[f"{day}_{t}_R1", s]
                r2 = x[f"{day}_{t}_R2", s]

                # Etablierung der Solver Variable "par"
                par = model.NewBoolVar(f"par_{day}_{t}_{s}")
                # Vergleich von Raum1 und Raum2 im aktuellen Slot
                model.AddMinEquality(par, [r1, r2])          # 1, wenn s beide Räume hat
                parallel.append(par)

                # Etablierung der Solver-Variable "used"
                used[t] = model.NewBoolVar(f"used_{day}_{t}_{s}")
                # Vergleich von Raum1 und Raum2 im aktuellen Slot
                model.AddMaxEquality(used[t], [r1, r2])      # 1, wenn s mind. einen Raum hat

            # Reihenfolgen Thema ....
            for t1, t2 in zip(slots, slots[1:]):             # (A,B), (B,C), (C,D)
                seq = model.NewBoolVar(f"seq_{day}_{t1}{t2}_{s}")
                # Rückschau auf used-Variable
                model.AddMinEquality(seq, [used[t1], used[t2]])   # 1, wenn s beide Zeitslots hat
                sequential.append(seq)

        # g = 1: allowed = false -> verboten
        if not rule["parallel"]["allowed"]:
            model.Add(sum(parallel) == 0)
        if not rule["sequential"]["allowed"]:
            model.Add(sum(sequential) == 0)

        # g >= 3: parallel bevorzugt (6 > 3)
        if rule["parallel"]["preference"] > rule["sequential"]["preference"]:
            model.Add(sum(parallel) >= sum(sequential))                                     # (a)
            model.Add(sum(parallel) >= math.floor(bounds_week[s][0] * PAR_SHARE / 2))       # (b)

In [27]:
# bereitet Lösung hinsichtlich Popularität und Struktur für Darstellung auf

def pop_and_struc(solution):
    rows = []

    # Jeden Studiengang einzeln betrachten
    for s in sorted(set(solution.values())):

        # Alle Slots dieses Studiengangs sammeln
        own = []
        for v, sg in solution.items():
            if sg == s:
                own.append(v)

        # Die belegten Tage und Zeitslots speichern
        times = set()
        for v in own:
            teile = v.split("_")
            tag = teile[0]
            slot = teile[1]
            times.add((tag, slot))

        # Gleichzeitige Prüfungen in beiden Räumen zählen
        parallel = 0
        for d in weekdays:
            for t in slots:
                if (solution[f"{d}_{t}_R1"] == s and
                    solution[f"{d}_{t}_R2"] == s):
                    parallel += 1

        # Direkt aufeinanderfolgende Prüfungen zählen
        sequential = 0
        for d in weekdays:
            for i in range(len(slots) - 1):
                t1 = slots[i]
                t2 = slots[i + 1]

                if (d, t1) in times and (d, t2) in times:
                    sequential += 1

        # Durchschnittliche Popularität berechnen
        durchschnitt = sum(pop[v] for v in own) / len(own)

        # Ergebnisse speichern
        rows.append({
            "SG": s,
            "Slots": len(own),
            "mittl. Popularität": round(durchschnitt, 2),
            "parallel": parallel,
            "aufeinanderfolgend": sequential
        })

    # Ergebnisse als Tabelle zurückgeben
    return pd.DataFrame(rows).set_index("SG")

### 2.6 Übergangsregel

Wenn ein Studiengang den letzten Slot eines Tages hat, so darf er am nächsten Tag nicht den ersten Slot haben.

$$x_{s,d,D,r_1} + x_{s,d+1,A,r_2} \le 1$$
für alle $d \in \{Mo, Di, Mi, Do\}$ und $r_1, r_2 \in \{R1, R2\}$

Heißt: Slot D am Tag $d$ und Slot A am Tag $d+1$ dürfen nicht beide demselben Studiengang gehören, egal in welchem Raum.

- 4 Tageswechsel × 4 Raumkombinationen = 16 Constraints pro Studiengang
- Umsetzung: `add_transition_constraints`
- Kontrolle am fertigen Plan: `check_transitions` (siehe 4.)


- Parallele Nutzung beider Räume durch denselben Studiengang soll nur dann zulässig oder erforderlich sein, wenn dies aufgrund seiner Kursgruppenzahl sinnvoll ist.

In [ ]:
def add_transition_constraints(model, x, programs):
    for s in programs:
        for day, next_day in zip(weekdays, weekdays[1:]):        # (Mo,Di) ... (Do,Fr)
            for r_late in rooms:
                for r_early in rooms:
                    late = x[f"{day}_D_{r_late}", s]
                    early = x[f"{next_day}_A_{r_early}", s]
                    model.Add(late + early <= 1)                  # nicht beides


# Kontrolle am fertigen Plan (nach Colins Prüfung), unabhängig vom Solver
def check_transitions(solution):
    errors = []
    for s in sorted(set(solution.values())):
        for day, next_day in zip(weekdays, weekdays[1:]):
            late = any(solution[f"{day}_D_{r}"] == s for r in rooms)
            early = any(solution[f"{next_day}_A_{r}"] == s for r in rooms)
            if late and early:
                errors.append(f"{s}: letzter Slot am {day}, erster Slot am {next_day}")
    return errors

# 3. Implementierung des Constraint-Modells

## CSP System

Umsetzung mit OR-Tools (CP-SAT)

- Variable = Tag + Slot + Raum
- Wert = Studiengang

Darstellung der Variablen durch Boolesche Variablen $x_{v,s} \in \{0,1\}$:

$x_{v,s} = 1$ heißt „Slot $v$ gehört Studiengang $s$“.


In [ ]:
# CSP System

from ortools.sat.python import cp_model

def build_csp(week):
    week_df = exam_weeks[exam_weeks["exam_week"] == week]
    programs = week_df["study_program"].tolist() # nur Studiengänge dieser Woche (--> quasi Domain!)

    model = cp_model.CpModel()
    x = {
        (v, s): model.NewBoolVar(f"{v}_{s}")
        for v in variables # 'variables' kommt von der Definition am Anfang des Dokuments
        for s in programs
    }
    # Beispiel: ('Mo_A_R1', 'ES'): Mo_A_R1_ES(0..1)

    # In einem Slot kann nur genau 1 Studiengang enthalten sein (umgkehrt bereits durch Modell abgesichert)
    for v in variables:
        model.AddExactlyOne(
            x[v, s] for s in programs
        )

    # Bedarfe

    # -> Bedarf pro Woche
    week_demand = demand_bounds(week_df)
    for s, (lo, hi) in week_demand.items():
        model.AddLinearConstraint(
            sum(x[v, s] for v in variables), # Ergibt lineare Summe: x_{1,s} + x_{2,s} + ... +  x_{n,s} -> wird erst später vom Solver ausgewertet !!
            lo, # Constraint garantiert Einhaltung der Untergrenze
            hi  # ... und Obergrenze
        )
    # => Solver weiß dann, dass Lösung für Lineare Gleichung zwischen LO und HI liegen muss

    # -> Bedarf pro Tag (analog Woche)
    for day in weekdays:
        day_vars = [v for v in variables if v.startswith(day)] # nur Variablen des betroffenen Tages heranziehen
        day_demand = demand_bounds(week_df, SLOTS_PER_DAY)

        for s, (lo, hi) in day_demand.items():
            model.AddLinearConstraint(
                sum(x[v, s] for v in day_vars),
                lo,
                hi
            )

    # Popularität
    add_popularity_constraints(model, x, programs) # siehe unten

    # Parallele / Aufeinanderfolgende Termine
    add_structure_constraints(model, x, week_df, programs) # siehe unten

    # Kein Übergang spät -> früh (4.5)
    add_transition_constraints(model, x, programs) # siehe unten


    return model, x


def solve(model, x):
    solver = cp_model.CpSolver()
    status = solver.Solve(model)
    if status not in (cp_model.OPTIMAL, cp_model.FEASIBLE):
        return None
    return {v: s for (v, s), var in x.items() if solver.Value(var)}

## 4. Lösungen der Prüfungswochen und Validierung
Für jede Woche wird das CSP gelöst, der Belegungsplan ausgegeben und die Constraints überprüft.

In [ ]:

def show_plan(solution):
    rows = [
        {
            "Tag": v.split("_")[0],
            "Slot": v.split("_")[1],
            "Raum": v.split("_")[2],
            "SG": s # study_program
        }
        for v, s in solution.items()
    ]

    #plan = pd.DataFrame(rows)
    #print (plan)
    #     Tag   Slot Raum  SG
    # 0   Mo    A    R1    INF
    # 1   Mo    A    R2    DS

    plan = pd.DataFrame(rows).pivot(
        index=["Slot", "Raum"],
        columns="Tag",
        values="SG"
    )
    return plan[weekdays]

def slots_per_day(solution):
    rows = [
        {
            "Tag": v.split("_")[0],
            "SG": s # study_program
        }
        for v, s in solution.items()
    ]

    df = pd.DataFrame(rows)
    return pd.crosstab(
        df["SG"],
        df["Tag"]
    )[weekdays]

solutions = {}
for week in weeks:

    print("")
    print("========================================================")
    print(f"                    WEEK: {week}")
    print("========================================================")
    print("")

    solutions[week] = solve(*build_csp(week))
    # e.g. {'Mo_A_R1': 'INF', 'Mo_A_R2': 'DS', 'Mo_B_R1': 'MB', ........ }

    print("Week plan:")
    display(show_plan(solutions[week]))
    print("")

    print("Week demand:")
    print(week, pd.Series(solutions[week]).value_counts().to_dict()) # print how often each course de-facto occurs in this week (cmp. demand_bounds())
    print("")

    print("Day demand:")
    display(slots_per_day(solutions[week]))
    print("")

    print("Popularität und Struktur:")
    display(pop_and_struc(solutions[week]))


                    WEEK: Woche 1

Week plan:


Tag        Mo   Di  Mi   Do  Fr
Slot Raum                      
A    R1    DS   MT  DS   MB  DS
     R2    MT   MT  ES  INF  DS
B    R1    MB   ES  DS   MT  MB
     R2    MB   DS  MB   MB  MB
C    R1    MB   MB  MB   DS  MT
     R2    MB   MB  MB   DS  MT
D    R1    ES  INF  MT   ES  ES
     R2    ES   MB  MT   MB  MB


Week demand:
Woche 1 {'MB': 16, 'DS': 8, 'MT': 8, 'ES': 6, 'INF': 2}

Day demand:


Tag,Mo,Di,Mi,Do,Fr
SG,,,,,
DS,1,1,2,2,2
ES,2,1,1,1,1
INF,0,1,0,1,0
MB,4,3,3,3,3
MT,1,2,2,1,2



Popularität und Struktur:


...

Tag         Mo   Di   Mi   Do   Fr
Slot Raum                         
A    R1     MB   ET   ET   DS   DS
     R2     MB  WIW   DS   DS  WIW
B    R1     MB   DS  WIW   ET   DS
     R2     MB  WIW   MB   MB   MB
C    R1    WIW   MB   MB  WIW  WIW
     R2    WIW  WIW   DS  WIW   ET
D    R1     DS   MB   MB   MB   MB
     R2     DS   MB  WIW   MB   MB


Week demand:
Woche 3 {'MB': 16, 'WIW': 11, 'DS': 9, 'ET': 4}

Day demand:


Tag,Mo,Di,Mi,Do,Fr
SG,,,,,
DS,2,1,2,2,2
ET,0,1,1,1,1
MB,4,3,3,3,3
WIW,2,3,2,2,2



Popularität und Struktur:


,Slots,mittl. Popularität,parallel,aufeinanderfolgend
SG,,,,
DS,9,6.89,2,1
ET,4,6.75,0,0
MB,16,6.69,5,4
WIW,11,6.36,2,2


In [ ]:
# Kontrolle: enthalten die Lösungen aller Wochen noch Übergänge spät -> früh?

for week in weeks:
    model, x = build_csp(week)
    sol = solve(model, x)
    if sol is None:
        print(f"{week}: keine Lösung")
        continue
    errors = check_transitions(sol)
    print(f"{week}: {len(errors)} Übergänge spät -> früh", errors)

## 5. Analyse des Lösungsraums
Für Woche 1 werden 5 zulässige Lösungen bestimmt. Jede gefundene Lösung wird per Blocking-Constraint (Blocking Clause) ausgeschlossen: Mindestens eine Zuordnung muss sich von ihr unterscheiden.

In [ ]:
# Mehrere Lösungen bestimmen (jede gefundene Lösung per Blocking-Constraint ausschließen)

def solve_reproducible(model, x, seed=42):
    # wie solve(), aber mit einem Such-Thread und festem Seed -> gleiche Lösungen bei jedem Lauf
    solver = cp_model.CpSolver()
    solver.parameters.num_workers = 1
    solver.parameters.random_seed = seed
    status = solver.Solve(model)
    if status not in (cp_model.OPTIMAL, cp_model.FEASIBLE):
        return None
    return {v: s for (v, s), var in x.items() if solver.Value(var)}

def enumerate_solutions(week, k=5):
    found = []
    for i in range(k):
        model, x = build_csp(week)

        # jede bereits gefundene Lösung ausschließen:
        # von ihren 40 Zuordnungen x[v, s] = 1 darf höchstens 39 wieder gleich sein
        for prev in found:
            model.Add(
                sum(x[v, s] for v, s in prev.items()) <= len(prev) - 1
            )

        sol = solve_reproducible(model, x)
        if sol is None:
            break
        found.append(sol)
    return found

week = "Woche 1"
sols = enumerate_solutions(week, k=5)
print(f"{len(sols)} unterschiedliche Lösungen für {week}\n")

for i, sol in enumerate(sols):
    print(f"--- Lösung {i+1} ---")
    display(show_plan(sol))

5 unterschiedliche Lösungen für Woche 1

--- Lösung 1 ---


Tag        Mo  Di   Mi   Do  Fr
Slot Raum                      
A    R1    MB  DS   MB  INF  MB
     R2    MB  MB   ES   ES  MT
B    R1    MB  MB   MB   DS  DS
     R2    MT  MB   MB   MB  DS
C    R1    DS  ES   DS   MT  MB
     R2    MT  DS   DS   MT  ES
D    R1    ES  MT  INF   MB  MB
     R2    ES  MT   MT   MB  MB

--- Lösung 2 ---


Tag        Mo   Di  Mi   Do  Fr
Slot Raum                      
A    R1    ES   MB  MT   DS  DS
     R2    ES   MB  MB   DS  MB
B    R1    DS  INF  DS   MB  MT
     R2    MB   MB  DS   ES  MT
C    R1    MB   DS  MB   MT  ES
     R2    MB   ES  MB  INF  DS
D    R1    MT   MT  MT   MB  MB
     R2    MT   MB  ES   MB  MB

--- Lösung 3 ---


Tag        Mo   Di   Mi  Do  Fr
Slot Raum                      
A    R1    DS  INF   MB  MB  MB
     R2    ES   MB   MB  DS  MB
B    R1    MB   MT   ES  MT  DS
     R2    MT   MB   ES  MT  DS
C    R1    MT   ES   MB  MB  MT
     R2    MB   MB   DS  MB  MT
D    R1    MB   DS   MT  DS  ES
     R2    MB   DS  INF  ES  MB

--- Lösung 4 ---


Tag         Mo  Di  Mi  Do   Fr
Slot Raum                      
A    R1     MB  ES  DS  DS   MT
     R2     ES  MB  MB  MT   MB
B    R1    INF  MB  MT  MB   DS
     R2     DS  MB  MT  DS   DS
C    R1     MT  DS  MB  MT   ES
     R2     MT  DS  MB  ES   MB
D    R1     MB  MT  ES  MB  INF
     R2     MB  MB  ES  MB   MB

--- Lösung 5 ---


Tag         Mo   Di  Mi  Do  Fr
Slot Raum                      
A    R1     MB   MT  MB  MB  MT
     R2     ES   MT  MB  MB  MB
B    R1     DS   ES  MT  MT  DS
     R2     DS   MB  MT  MB  ES
C    R1    INF  INF  DS  DS  MB
     R2     MT   MB  ES  DS  DS
D    R1     MB   DS  MB  ES  MT
     R2     MB   MB  MB  ES  MB

In [ ]:
# Wie stark unterscheiden sich die Lösungen?

# Anzahl Slots, die zwei Lösungen unterschiedlich belegen
def slot_diff(a, b):
    return sum(a[v] != b[v] for v in variables)

# wie slot_diff, aber R1 und R2 als gleichwertig betrachtet:
# zählt Zeitpunkte (Tag, Slot), an denen die beiden Räume insgesamt anders verteilt sind
def time_diff(a, b):
    return sum(
        sorted([a[f"{d}_{t}_R1"], a[f"{d}_{t}_R2"]]) != sorted([b[f"{d}_{t}_R1"], b[f"{d}_{t}_R2"]])
        for d in weekdays for t in slots
    )

labels = [f"L{i+1}" for i in range(len(sols))]
print("Unterschiedliche Slots (von 40):")
display(pd.DataFrame([[slot_diff(a, b) for b in sols] for a in sols], index=labels, columns=labels))

print("Unterschiedliche Zeitpunkte, Räume getauscht ignoriert (von 20):")
display(pd.DataFrame([[time_diff(a, b) for b in sols] for a in sols], index=labels, columns=labels))

In [ ]:
# Welche Zuordnungen variieren? Pro Slot alle Studiengänge, die in den Lösungen dort vorkommen

variation = {v: sorted({sol[v] for sol in sols}) for v in variables}
constant = [v for v, s in variation.items() if len(s) == 1]
print(f"Slots mit immer gleicher Belegung: {len(constant)} von {N_SLOTS}  {constant}")

var_plan = show_plan({v: "/".join(s) for v, s in variation.items()})
display(var_plan)

# Was bleibt gleich, was ändert sich? Kennzahlen je Lösung nebeneinander
kpis = pd.concat(
    {f"L{i+1}": pop_and_struc(sol) for i, sol in enumerate(sols)},
    axis=1
)
for col in ["Slots", "mittl. Popularität", "parallel", "aufeinanderfolgend"]:
    print(col)
    display(kpis.xs(col, axis=1, level=1))

In [ ]:
# Wie viele Lösungen gibt es? Zählen bis zum Zeitlimit (= Untergrenze)

class SolutionCounter(cp_model.CpSolverSolutionCallback):
    def __init__(self):
        super().__init__()
        self.count = 0
    def on_solution_callback(self):
        self.count += 1

for w in weeks:
    model, x = build_csp(w)
    solver = cp_model.CpSolver()
    solver.parameters.enumerate_all_solutions = True
    solver.parameters.max_time_in_seconds = 5
    counter = SolutionCounter()
    status = solver.Solve(model, counter)
    vollstaendig = status == cp_model.OPTIMAL
    print(f"{w}: {counter.count} Lösungen in 5 s gefunden"
          + ("" if vollstaendig else " (abgebrochen, es gibt mehr)"))

In [ ]:
# Kann jeder Slot grundsätzlich jeden Studiengang bekommen?
# Für jedes Paar (Slot, Studiengang): gibt es irgendeine zulässige Lösung mit x[v, s] = 1?

def possible_assignments(week):
    possible = set()
    model, x = build_csp(week)
    for (v, s), var in x.items():
        if (v, s) in possible:
            continue                        # schon in einer früheren Lösung gesehen
        m = model.Clone()
        m.Add(m.GetBoolVarFromProtoIndex(var.Index()) == 1)
        sol = solve(m, x)
        if sol is not None:
            possible |= set(sol.items())    # alle Zuordnungen dieser Lösung sind möglich
    return possible, len(x)

possible, total = possible_assignments(week)
print(f"{week}: {len(possible)} von {total} Zuordnungen (Slot, Studiengang) kommen in mindestens einer Lösung vor")

### Auswertung (Woche 1)

**Welche Constraints reduzieren den Lösungsraum besonders deutlich?**
- Der Lösungsraum bleibt groß: über 600 Lösungen je Woche in 5 s (Untergrenze), jede der 200 Zuordnungen (Slot, Studiengang) ist in mindestens einer Lösung möglich.
- Die Constraints legen also nicht fest, *wo* ein Studiengang prüft, sondern *wie viel* und *wie verteilt*. Am deutlichsten wirken Tagesbedarf (max. Slots pro Tag) und Popularität (siehe 6).

**Welche Auswirkungen haben Bedarf und Popularitäts-Fairness?**
- Bedarf: Slotzahlen in allen Lösungen gleich (6/2/16/8/8), in Woche 1 exakt festgelegt.
- Popularität: mittlere Popularität aller Studiengänge zwischen 6.25 und 7.12, immer im Band 6.15 – 7.15.

**Welche Zuordnungen variieren?**
- Je zwei Lösungen unterscheiden sich in 16–32 von 40 Slots, ohne Raumtausch in 12–20 von 20 Zeitpunkten. Die Unterschiede sind also nicht nur R1 ↔ R2.
- Nur `Do_B_R1` und `Fr_C_R1` sind in allen 5 Lösungen gleich, das ist aber Zufall der Stichprobe (200 von 200 möglich).
- Gleich bleiben: Slotzahlen, INF nie parallel/aufeinanderfolgend, ES immer genau 1 Parallelpaar.
- Variieren: Lage der Slots, mittlere Popularität, Paare bei MB (4–5 parallel, 4–5 aufeinanderfolgend) sowie MT/DS (2–3 parallel).

## 6. Experiment mit verändertem Constraint

Verändert wird die Toleranz $\tau$ der Popularitäts-Fairness (`POP_TOL`, siehe 2.4). Im Modell gilt $\tau = 0.5$.

Für Woche 1 wird $\tau$ einmal nach oben bis 2.0 und einmal schrittweise nach unten (0.25 / 0.15 / 0.1 / 0) verändert. Alle anderen Constraints bleiben gleich.

**Hypothese:**
- **Bedarfsverteilung:** bleibt gleich. Die Slotzahlen hängen nur am Bedarf (2.3), nicht an der Popularität. In Woche 1 also immer 6/2/16/8/8.
- **Zulässige Zuordnungen:**
  - $\tau$ nach oben: das Band wird breiter, Studiengänge dürfen im Mittel deutlich bessere oder schlechtere Slots bekommen. Bei 2.0 (Band 4.65 – 8.65) greift der Constraint praktisch nicht mehr.
  - $\tau$ nach unten: das Band wird enger. INF trifft es zuerst, weil bei 2 Slots der Mittelwert immer ein Vielfaches von 0.5 ist. 7.0 ist nur für $\tau \ge 0.35$ zulässig, 6.5 nur für $\tau \ge 0.15$.
- **Lösungsraum:** wächst mit größerem $\tau$, schrumpft mit kleinerem. Unter $\tau = 0.15$ wird Woche 1 unlösbar (INF).

**Gemessen wird pro $\tau$:**
- lösbar ja/nein
- Slots je Studiengang
- kleinste und größte erreichbare mittlere Popularität von INF und MB (Solver einmal mit Minimize, einmal mit Maximize – nur zur Analyse, das Modell selbst optimiert nicht)
- Lösungen in 3 s (Untergrenze, nur als Richtwert)
- Lösbarkeit aller drei Wochen

`POP_TOL` wird danach wieder auf 0.5 gesetzt.

In [ ]:
# Experiment: Toleranz τ der Popularitäts-Fairness hoch- und runterskalieren

class SolutionCounter(cp_model.CpSolverSolutionCallback):
    def __init__(self):
        super().__init__()
        self.count = 0
    def on_solution_callback(self):
        self.count += 1

def count_solutions(model, seconds=3):
    # Lösungen aufzählen, bis das Zeitlimit erreicht ist (Untergrenze, nur als Richtwert)
    solver = cp_model.CpSolver()
    solver.parameters.enumerate_all_solutions = True
    solver.parameters.max_time_in_seconds = seconds
    counter = SolutionCounter()
    status = solver.Solve(model, counter)
    return counter.count, status == cp_model.OPTIMAL

def popularity_range(week, s):
    # kleinste und größte erreichbare mittlere Popularität von Studiengang s
    result = []
    for sense in ("min", "max"):
        model, x = build_csp(week)
        total = sum(pop[v] * x[v, s] for v in variables)
        model.Minimize(total) if sense == "min" else model.Maximize(total)
        sol = solve_reproducible(model, x)
        own = [v for v, sg in sol.items() if sg == s]
        result.append(sum(pop[v] for v in own) / len(own))
    return result

def tau_experiment(taus, week="Woche 1"):
    global POP_TOL
    mean_pop = popTotal / N_SLOTS
    rows = []
    for tau in taus:
        POP_TOL = tau                                # wird in add_popularity_constraints gelesen
        sol = solve_reproducible(*build_csp(week))
        row = {
            "τ": tau,
            "erlaubtes Band": f"{mean_pop - tau:.2f} – {mean_pop + tau:.2f}",
            "lösbar": sol is not None,
        }
        if sol is not None:
            kpi = pop_and_struc(sol)
            n, complete = count_solutions(build_csp(week)[0])
            inf_min, inf_max = popularity_range(week, "INF")
            mb_min, mb_max = popularity_range(week, "MB")
            row.update({
                "Slots ES/INF/MB/MT/DS": "/".join(str(kpi.loc[s, "Slots"]) for s in ["ES", "INF", "MB", "MT", "DS"]),
                "Ø Pop. INF möglich": f"{inf_min:.2f} – {inf_max:.2f}",
                "Ø Pop. MB möglich": f"{mb_min:.2f} – {mb_max:.2f}",
                "Lösungen in 3 s": f"{n}" + ("" if complete else "+"),
            })
        rows.append(row)
    return pd.DataFrame(rows).set_index("τ").fillna("–")

tau_original = POP_TOL
try:
    print("τ nach oben (lockerer), Woche 1:")
    display(tau_experiment([0.5, 1.0, 2.0]))

    print("τ nach unten (strenger), Woche 1:")
    display(tau_experiment([0.5, 0.25, 0.15, 0.1, 0.0]))

    print("Lösbarkeit aller Wochen:")
    status = {}
    for tau in [2.0, 1.0, 0.5, 0.25, 0.15, 0.1, 0.0]:
        POP_TOL = tau
        status[tau] = {w: ("lösbar" if solve_reproducible(*build_csp(w)) else "unlösbar") for w in weeks}
    display(pd.DataFrame(status).T.rename_axis("τ"))
finally:
    POP_TOL = tau_original                           # Originalwert für alle folgenden Zellen

#### Wahl der Toleranz $\tau = 0.5$
Wahl von Toleranz führt teils zu nicht lösbarem Solving.....
| $\tau$ | Woche 1 | Woche 2 | Woche 3 |
|---|---|---|---|
| 0 | unlösbar | unlösbar | unlösbar |
| 0.1 | unlösbar | lösbar | lösbar |
| 0.25 | lösbar | lösbar | lösbar |
| 0.5 | lösbar | lösbar | lösbar |

Woche 1 scheitert bei $\tau = 0.1$ am Studiengang "INF":
- Beim Teilen durch 1 oder 2 kommt entweder wieder Ganzzahl oder .5 heraus
- 6.65 +/- 0.1 führt hier zu ungültigem Intervall mit 6.55 bzw. 6.75
- Solver kann keine Lösung finden!

(siehe auch Abschnitt "9 Kritische Reflexion")

**Vergleich Hypothese / Ergebnis:**
- **Bedarfsverteilung:** bei jedem lösbaren $\tau$ 6/2/16/8/8 → bestätigt.
- **$\tau = 2.0$:** INF 5.00 – 8.50, MB 5.25 – 8.12. Der Fairness-Constraint begrenzt kaum noch, die Grenzen kommen von den anderen Constraints → bestätigt. Deshalb bleibt es bei $\tau = 0.5$.
- **$\tau$ nach unten:** INF ab 0.25 nur noch 6.5 (7.0 fällt unter 0.35 weg), MB von 6.19 – 7.12 auf 6.56 – 6.75 bei 0.15 → bestätigt.
- **Lösungsraum:** Lösungen in 3 s von mehreren Tausend (2.0) auf wenige Hundert (0.15) → Tendenz bestätigt, die Zahlen schwanken je Lauf.
- **Grenze:** 0.15 lösbar, 0.1 unlösbar → genau wie vorhergesagt. Woche 2 und 3 bleiben bis 0.1 lösbar, bei 0 sind alle Wochen unlösbar.

## 7. Unlösbarkeit und Konfliktanalyse

**Verschärfung:** Große Studiengänge ($g_s \ge 3$) sollen nicht nur die Hälfte, sondern **drei Viertel** ihrer Slots als Parallelpaare bekommen (`PAR_SHARE` von 0.5 auf 0.75, siehe 2.5).

Fachlich nachvollziehbar: parallel hat im Datensatz die höchste Präferenz (6). Mehr parallele Termine heißt mehr Kursgruppen gleichzeitig.

Für Woche 1 heißt das mit Formel (b) aus 2.5: ES 2 Paare, MB 6, MT 3, DS 3.

Jede Bedingung ist für sich genommen erfüllbar. Ob alle zusammen erfüllbar sind, zeigt die Rechnung. `PAR_SHARE` wird danach wieder auf 0.5 gesetzt.

In [ ]:
# 7. Unlösbarkeit: Parallel-Anteil für große Studiengänge von 0.5 auf 0.75 verschärfen

par_share_original = PAR_SHARE

def solvable_weeks():
    return {w: ("lösbar" if solve_reproducible(*build_csp(w)) else "unlösbar") for w in weeks}

try:
    # --- (1) Verschärfung: mind. 3/4 der Slots in Parallelpaaren
    PAR_SHARE = 0.75
    print("PAR_SHARE = 0.75:", solvable_weeks())

    # geforderte Parallelpaare je Studiengang (g >= 3) in Woche 1
    week_df = exam_weeks[exam_weeks["exam_week"] == "Woche 1"]
    lo_week = {s: lo for s, (lo, hi) in demand_bounds(week_df).items()}
    lo_day = {s: lo for s, (lo, hi) in demand_bounds(week_df, SLOTS_PER_DAY).items()}
    print("geforderte Paare (Woche 1):",
          {s: math.floor(lo_week[s] * PAR_SHARE / 2) for s in ["ES", "MB", "MT", "DS"]})
    print("ES: Slots pro Woche =", lo_week["ES"], "| mind. Slots pro Tag =", lo_day["ES"])
finally:
    PAR_SHARE = par_share_original

In [ ]:
# (2) Kleinste Konfliktmenge: nur die drei Constraints für ES, sonst nichts
# C3: ES hat genau 6 Slots | C4: ES an jedem Tag mind. 1 Slot | C7b: ES mind. 2 Parallelpaare

def es_model(use_c3=True, use_c4=True, use_c7b=True):
    model = cp_model.CpModel()
    es = {v: model.NewBoolVar(v) for v in variables}           # 1 = Slot v gehört ES
    if use_c3:
        model.Add(sum(es.values()) == 6)
    if use_c4:
        for day in weekdays:
            model.Add(sum(es[v] for v in variables if v.startswith(day)) >= 1)
    if use_c7b:
        pairs = []
        for day in weekdays:
            for t in slots:
                p = model.NewBoolVar(f"par_{day}_{t}")
                model.AddMinEquality(p, [es[f"{day}_{t}_R1"], es[f"{day}_{t}_R2"]])
                pairs.append(p)
        model.Add(sum(pairs) >= 2)
    solver = cp_model.CpSolver()
    return solver.StatusName(solver.Solve(model))

print("C3 + C4 + C7b    :", es_model())
print("ohne C3 (6 Slots):", es_model(use_c3=False))
print("ohne C4 (je Tag) :", es_model(use_c4=False))
print("ohne C7b (Paare) :", es_model(use_c7b=False))

**Ergebnis:** Woche 1 wird unlösbar, Woche 2 und 3 bleiben lösbar.

**1. Welche Constraints stehen in Konflikt?**

Drei Constraints, alle für ES in Woche 1:
- **Wochenbedarf:** ES hat genau 6 Slots (2.3.1)
- **Tagesbedarf, Untergrenze:** ES an jedem der 5 Tage mind. 1 Slot (2.3.2)
- **Mindest-Parallelpaare (b):** ES mind. $\lfloor 6 \cdot 0.75 / 2 \rfloor = 2$ Paare (2.5)

Die zweite Zelle prüft genau diese drei Constraints allein, ohne die übrigen: zusammen `INFEASIBLE`, lässt man einen weg, ist es wieder lösbar. Die Menge ist also minimal. Popularität, Übergangsregel und die anderen Studiengänge sind für den Konflikt nicht nötig.

**2. Warum gibt es keine Lösung?**
- 5 der 6 ES-Slots gehen an die Tagesuntergrenze (1 pro Tag). Es bleibt 1 Slot übrig.
- Die Verteilung ist also zwingend 2/1/1/1/1: nur ein Tag mit 2 Slots.
- Ein Parallelpaar braucht 2 Slots am selben Tag zur selben Zeit → höchstens 1 Paar möglich.
- Gefordert sind 2 → Widerspruch.

ES gibt es nur in Woche 1. Die großen Studiengänge in Woche 2 und 3 haben mehr Slots pro Tag und damit genug Doppeltage für ihre Paare.

In [ ]:
# (3) Reparatur: PAR_SHARE nur so weit senken, dass ES 1 Paar statt 2 braucht
# floor(6 * PAR_SHARE / 2) = 1  gilt für PAR_SHARE < 2/3  ->  0.65

try:
    PAR_SHARE = 0.65
    print("PAR_SHARE = 0.65:", solvable_weeks())

    sol = solve_reproducible(*build_csp("Woche 1"))
    display(show_plan(sol))
    display(pop_and_struc(sol))
    print("Übergänge spät -> früh:", check_transitions(sol))
finally:
    PAR_SHARE = par_share_original

# Grenze prüfen: ab 2/3 braucht ES wieder 2 Paare
try:
    for share in [0.66, 0.67]:
        PAR_SHARE = share
        print(f"PAR_SHARE = {share}:", solvable_weeks())
finally:
    PAR_SHARE = par_share_original

**3. Kleinste Änderung**

`PAR_SHARE` nicht auf 0.5 zurücksetzen, sondern nur so weit senken, dass ES 1 Paar statt 2 braucht:
$\lfloor 6 \cdot PAR\_SHARE / 2 \rfloor = 1$ gilt für $PAR\_SHARE < 2/3$. Gewählt: **0.65**.

- Alle drei Wochen sind wieder lösbar (Plan und Kennzahlen für Woche 1 oben, ES hat 1 Paar).
- Die Grenze ist scharf: 0.66 lösbar, 0.67 unlösbar.
- Das passt zu 2.5: ein Anteil von 2/3 ist wegen ES nicht erfüllbar.

Andere Reparaturen wären, einen der beiden anderen Constraints für ES zu lockern (z. B. ES darf an einem Tag 0 Slots haben). Das würde aber eine fachliche Anforderung (Bedarf bzw. jeder Tag) aufweichen. Die Änderung an `PAR_SHARE` nimmt nur einen Teil der Verschärfung zurück.

## 8. Analyse des Suchverfahrens

CP-SAT ist ein Constraint-Solver, der intern auf einem SAT-Solver aufbaut (Lazy Clause Generation) [1]:
- **Propagation:** Nach jeder Entscheidung werden unmögliche Werte gestrichen. Beispiel: Bekommt ein Slot einen Studiengang, sind alle anderen Studiengänge für diesen Slot raus.
- **Lernen aus Konflikten:** Führt eine Entscheidung zu einem Widerspruch, merkt sich der Solver die Ursache und vermeidet sie später.
- **Mehrere Such-Threads** mit unterschiedlichen Strategien laufen parallel.

**In unserem Modell** (Woche 1: 40 Slots × 5 Studiengänge = 200 Boolesche Variablen $x_{v,s}$):
- `AddExactlyOne` und Übergangsregel betreffen nur wenige Variablen und greifen sofort.
- Bedarf und Popularität sind Summen über viele Slots und greifen erst richtig, wenn schon viele Slots belegt sind.
- `par` / `used` / `seq` sind Hilfsvariablen aus $x$, keine eigenen Entscheidungen.

**Beobachtungen:**
- Jede Woche ist in Bruchteilen einer Sekunde gelöst. Auch Unlösbarkeit (Abschnitt 7) wird schnell erkannt.
- Durch die parallelen Threads kann jeder Lauf eine andere Lösung liefern. Deshalb wird in den Abschnitten 5–7 mit einem Thread und festem Seed gelöst (`solve_reproducible`).

[1] L. Perron, F. Didier, S. Gay: *The CP-SAT-LP Solver (Invited Talk)*. CP 2023, LIPIcs Bd. 280, S. 3:1–3:2. https://doi.org/10.4230/LIPIcs.CP.2023.3

## 9. Kritische Reflexion

### Welche Modellierungsentscheidung hatte den größten Einfluss auf Ihr CSP?

- **Dediziertes CSP pro Woche** (Reduziert die Problemgröße deutlich)
- **Popularitäts-Fairness mit Toleranz $\tau = 0.5$** (definiert einen engen  Korridor für die Slot-Auswahl)


### Wie haben Sie “faire Verteilung” bezüglich Bedarf und Popularität operationalisiert?

Über **Constraints mit Intervallgrenzen**

- **Bedarf:**
  - Ermittlung der Verhältnisses von Bedarfseinheiten zu Gesamtgruppenanzahl
  - Verwendung dieses Wertes für anteilige Berechnung an den 40- bzw. 8 Slots pro Woche bzw. Tag
  - Ab- bzw. Aufrunden für Ganzzahligkeit
  - Erhebung eines Constraints für Einhaltung des konkreten Bedarfes innerhalb der definierten Unter- bzw. Oberschranke
- **Popularität:**
  - Ermittlung der mittleren Popularität über die Woche hinweg für alle Studiengänge
  - Ermittlung der mittleren Popularität eines konkreten Studienganges
  - Erhebung eines Constraints der diese Popularität unter Berücksichtigung der Abweichung TAU in einem bestimmten Wertebereich hält (siehe Constraints)
  - Mittlere Popularität $\bar p_s$ pro Studiengang muss in $[\bar p - \tau, \bar p + \tau]$ mit $\bar p = 6.65$ und $\tau = 0.5$ liegen.



### Welche Auswirkungen hat die Wahl der Toleranzen auf die Lösbarkeit?

**Kleine Toleranzen führen zu Unlösbarkeit.**

Beispiel Woche 1, INF ($g_s = 1$, 2 Slots): Die mittleren Popularitäten können bei 2 Slots nur die Werte $\bar p_s \in \{6.0; 6.5; 7.0; 7.5; \dots\}$ (in 0.5-Schritten) annehmen.

Bei $\tau = 0.1$ ist das erlaubte Intervall $[6.55; 6.75]$ – kein erreichbarer Wert liegt darin. Erst bei $\tau = 0.5$ (Intervall $[6.15; 7.15]$) ist $6.5$ oder $7.0$ zulässig.
    

### Welche Constraints schränken den Lösungsraum besonders stark ein?
In absteigender Reihenfolge:
- **Popularitäts-Fairness**
- **Bedarfsgrenzen**
- parallele Belegungen für $g_s \ge 3$
- Übergangsregel
- Slot-Eindeutigkeit

### Gibt es redundante Constraints?

Ja, in zwei Fällen:
- Parallel/Sequentell Verbot dann unnötig, wenn ohnehin nur 1 Slot pro Tag
- Außerdem: Bei 2 Slots pro Tag kann nur entweder parallel oder sequentiell sein

Redundanz ist hier unkritisch, weil sie die Korrektheit nicht gefährdet, nur die Propagation minimal verlangsamt.


### Welche Grenzen besitzt das Modell gegenüber einer realen Prüfungsplanung?

Das Modell ist abstrahiert:

- Keine konkreten Raumkapazitäten, keine Studentenzahlen, keine fachlichen Gruppierungen.
- Räume werden als gleich gut behandelt -> Fachliche Zuordnung (PC-Räume etc.)
- Alle Kursgruppen eines Studiengangs werden als identisch behandelt.
- Keine Berücksichtigung von Prüfpersonen
- Popularitätswerte sind fest vorgegeben, nicht empirisch.
- Regeln gelten nur pro Woche
- Bei zu strengen Constraints kann es auch gar keine Lösungen geben, reale Planung würde versuchen immer Lösung zu finden

## 10. Dokumentation der GenAI-Nutzung

#### Bedarfs-Constraint
- Vorschlag der anteiligen Bedarfsermittlung mit lo/hi-Rundung + Grenzeinhaltung.
- Lösung gedanklich nachvollzogen. Verstanden, dass Verhältnis von Kursbedarf zu Gesamtkursbedarf gegen Wochen-Slots gerechnet genau den relativen Bedarf des Studiengangs ergibt

#### Popularitäts-Constraint
- Vorschlag der gemittelten Popularität + Toleranzwertverwendung.
- Lösung verstanden. Insbesondere hilfreich in Punkto Notwendigkeit der passenden Toleranz. Verständnis des Ganzzahl 0.5-Schritt Problems bei "INF" in Woche 1.
- Außerdem hilfreich war die Umformung der Ungleichung, da ja die ```ortools``` nicht mit Brüchen arbeiten können.

### Seq/Par Constraint
- Implementierungsvorschlag mit Hilfsvariablen und min/max Constraints.
- Algorithmik nachvollzogen. Insbesondere die AND/OR-Artigkeit des min/max Ansatzes.

### Allgemein
GenAI wurde zudem genutzt, um Kickstart-mäßig das richtige Rahmengerüst zu erhalten.
Außerdem zunächst mit klassischen python-Constraints gestartet. AI wies darauf hin, dass Performance-Probleme geben wird. Dann Umstieg auf ```ortools```.

## 11. Fazit
Das Notebook zeigt, dass sich die Raum-Zeit-Verteilung der zentralen Prüfungsslots als CSP modellieren und mit CP-SAT lösen lässt.

Die Modellierung berücksichtigt:

- **Bedarfsgerechtigkeit** (relative Kursgruppenanteile),
- **Popularitäts-Fairness** (additives Maß $p(d,t) = \text{Tag} + \text{Slot}$ mit Toleranz $\tau = 0.5$),
- **strukturelle Wünsche** (parallel/sequenziell je nach $g_s$),
- **organisatorische Regeln** (Slot-Eindeutigkeit, Übergangsregel, Tagesverteilung).

Für alle drei Prüfungswochen existieren zulässige Lösungen. Durch gezielte Constraint-Verschärfung lässt sich Unlösbarkeit provozieren, analysieren und wieder beheben. Die unabhängige Validierung bestätigt die Korrektheit der Lösungen.